# CNN with padding and max pooling on CIFAR-10

AIO 2026, Module 5, Lesson 1 (From MLP to CNN). This notebook builds the baseline CNN
from the lesson quiz: 3×3 convolutions with "same" padding and 2×2 max pooling.
It is compared with the 7×7 CNN in `CIFAR10_CNN.ipynb` and the MLP in `CIFAR10_MLP.ipynb`.

## 1. Data

In [10]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
from torchsummary import summary

transform = transforms.ToTensor()

train_set = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

## 2. Model

All convolutions use a 3×3 kernel, stride 1 and `padding=1`, so they keep H and W
unchanged: (32 − 3 + 2·1) / 1 + 1 = 32. Each `MaxPool2d(2, 2)` halves H and W, keeps
the number of channels and has no parameters.

| Layer | Kernels | Output shape (C, H, W) | Params |
|---|---|---|---|
| Input | | (3, 32, 32) | |
| Conv1 + ReLU | 32 × (3, 3, 3) | (32, 32, 32) | 896 |
| Conv2 + ReLU | 64 × (32, 3, 3) | (64, 32, 32) | 18,496 |
| MaxPool 2×2 | | (64, 16, 16) | 0 |
| Conv3 + ReLU | 128 × (64, 3, 3) | (128, 16, 16) | 73,856 |
| Conv4 + ReLU | 256 × (128, 3, 3) | (256, 16, 16) | 295,168 |
| MaxPool 2×2 | | (256, 8, 8) | 0 |
| Flatten | | 16,384 | |
| Linear + ReLU | | 256 | 4,194,560 |
| Linear | | 10 | 2,570 |
| **Total** | | | **4,585,546** |

In [11]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32,64,3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128,256,3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16384, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
        
        # He init cho cả Conv2d lẫn Linear
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Sanity check: kernel shape, output shape, parameter count

In [12]:
images, labels = next(iter(train_loader))   # lấy 1 batch để kiểm tra

model = CNN()
print(model.features[0].weight.shape)
print(model(images).shape)
summary(CNN(), (3, 32, 32), device="cpu")

torch.Size([32, 3, 3, 3])
torch.Size([64, 10])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 32, 32]             896
              ReLU-2           [-1, 32, 32, 32]               0
            Conv2d-3           [-1, 64, 32, 32]          18,496
              ReLU-4           [-1, 64, 32, 32]               0
         MaxPool2d-5           [-1, 64, 16, 16]               0
            Conv2d-6          [-1, 128, 16, 16]          73,856
              ReLU-7          [-1, 128, 16, 16]               0
            Conv2d-8          [-1, 256, 16, 16]         295,168
              ReLU-9          [-1, 256, 16, 16]               0
        MaxPool2d-10            [-1, 256, 8, 8]               0
          Flatten-11                [-1, 16384]               0
           Linear-12                  [-1, 256]       4,194,560
             ReLU-13                  [-1, 256]         

## 3. Training setup

In [13]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

torch.manual_seed(42)
model = CNN(10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

device: mps


In [14]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)  # forward
        loss = criterion(logits, labels) # compute loss

        optimizer.zero_grad()   # clear gradient
        loss.backward()         # backprop
        optimizer.step()        # update params

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        n += images.size(0)
    return total_loss / n, correct / n

def evaluate(model, loader, criterion, device):
    model.eval()                                   # chế độ đánh giá
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():                          # không tính gradient
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            n += images.size(0)
    return total_loss / n, correct / n

## 4. Training (10 epochs)

In [15]:
epochs = 10
for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"test loss {te_loss:.4f} acc {te_acc:.4f}")

Epoch  1 | train loss 1.4576 acc 0.4859 | test loss 1.1431 acc 0.5971
Epoch  2 | train loss 1.0572 acc 0.6326 | test loss 1.0003 acc 0.6529
Epoch  3 | train loss 0.8861 acc 0.6949 | test loss 0.9238 acc 0.6843
Epoch  4 | train loss 0.7730 acc 0.7366 | test loss 0.9016 acc 0.6898
Epoch  5 | train loss 0.6720 acc 0.7713 | test loss 0.8509 acc 0.7061
Epoch  6 | train loss 0.5753 acc 0.8023 | test loss 0.8811 acc 0.6977
Epoch  7 | train loss 0.4840 acc 0.8382 | test loss 0.8455 acc 0.7201
Epoch  8 | train loss 0.4026 acc 0.8673 | test loss 0.8839 acc 0.7167
Epoch  9 | train loss 0.3210 acc 0.8962 | test loss 0.9874 acc 0.7001
Epoch 10 | train loss 0.2486 acc 0.9229 | test loss 0.9272 acc 0.7214


## Results

Setup: Adam (lr = 1e-4), He (Kaiming uniform) initialisation, ReLU, batch size 64,
10 epochs, `torch.manual_seed(42)`. Accuracy reported at the final epoch.

| Model | Params | Train acc | Test acc |
|---|---|---|---|
| MLP, 1 hidden layer (Case 2) | 789,258 | 50.0% | 48.1% |
| CNN, 4 × Conv 7×7, no padding, no pooling | 4,211,146 | 87.9% | 63.6% |
| CNN, 4 × Conv 3×3, same padding, 2 × MaxPool (this notebook) | 4,585,546 | 92.3% | 72.1% |

### Observations
- Test accuracy rises from 63.6% (7×7 CNN) to 72.1%, and the parameter count matches
  the quiz baseline (4,585,546).
- The four 3×3 convolutions hold only 388,416 parameters (about 8.5% of the model).
  The first `Linear` layer holds 4,194,560 (about 91.5%).
- Both CNNs flatten a 256 × 8 × 8 tensor. The 7×7 model shrinks the image with
  unpadded convolutions; this model keeps full resolution through padding and shrinks
  it with two max-pooling steps.
- The model still overfits. Test loss is lowest at epoch 7 (0.846) and reaches 0.927 by
  epoch 10, while train accuracy climbs to 92.3%. An earlier run with a 128-unit hidden
  layer (2,486,986 params) reached 72.8% test accuracy with a much smaller train–test
  gap (13 points vs 20), which points to the large `Linear` layer as a main source of
  overfitting.
- Next experiments (one change at a time): average pooling instead of max pooling,
  `padding=0`, and doubling the channels to 64-128-256-512.